# Vaardigheden week 6: cijfers per persoon en basisfrequenties

Dit notebook hoort bij het vaardighedenspoor. Het gebruikt verzonnen steden, dorpen, tests en betalingen, zodat de vaardigheden los blijven van het project van deze week.

Je maakt van aantallen cijfers per persoon, betrapt een AI die door de verkeerde bevolking deelde, ziet waarom kleine plaatsen aan beide uiteinden van een ranglijst staan, vertaalt een verdubbeld risico naar mensen en rekent uit wat een positieve test betekent.

**Zo werk je ermee:** voer elke cel op volgorde uit. Voordat je een cel uitvoert die een ranglijst afdrukt, raad je wie bovenaan staat en schrijf je je gok op.

## 1. Een aantal, een basis en een periode

Een aantal zegt hoe vaak iets gebeurde. Een cijfer deelt dat aantal door een basis, het aantal mensen dat geteld had kunnen worden, over één periode. Zonder de basis kan een aantal je niet vertellen waar een probleem groter is.

De tabel hieronder bevat acht verzonnen steden. Hij heeft twee bevolkingskolommen: de volkstelling van 2015 en een schatting voor 2025. De griepgevallen zijn in 2025 gemeld.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(25)
pop_2015 = rng.integers(4_000, 60_000, size=8)          # volkstelling in 2015
growth = rng.uniform(0.9, 1.7, size=8)                   # sommige steden krompen, andere groeiden snel
pop_2025 = np.round(pop_2015 * growth).astype(int)       # geschatte bevolking in 2025
true_share = rng.uniform(0.008, 0.030, size=8)           # aandeel mensen dat griep krijgt
cases = rng.binomial(pop_2025, true_share)               # griepgevallen die de klinieken in 2025 meldden

towns = pd.DataFrame({"town": list("ABCDEFGH"), "pop_2015": pop_2015,
                      "pop_2025": pop_2025, "cases_2025": cases})
towns

## 2. Beschrijf de tabel en laat de jaren kloppen

Beantwoord de vier vragen uit week 1: rijen, kolommen, typen en ontbrekende waarden. Deze week komt er een vijfde vraag bij: **welke basis hoort bij het aantal?** De gevallen komen uit 2025, dus de basis moet de bevolking van 2025 zijn. De cel laat ook zien hoeveel elke stad sinds de volkstelling veranderde.

In [ ]:
print("rijen en kolommen:", towns.shape)
print()
print(towns.dtypes)
print()
print("ontbrekende waarden:", int(towns.isna().sum().sum()))
print()
change = (towns["pop_2025"] / towns["pop_2015"] - 1) * 100
print("verandering van de bevolking sinds de volkstelling van 2015, in procent:")
print(pd.Series(change.round(1).values, index=towns["town"]).to_string())

## 3. Lees code die een AI-assistent schreef

We vroegen een AI-assistent: *"Rangschik de steden naar hun griepcijfer in 2025, per 1.000 inwoners."* De assistent schreef de cel hieronder. Voer hem uit en beslis dan of de ranglijst klopt.

In [ ]:
# Geschreven door een AI-assistent
towns["ai_rate"] = towns["cases_2025"] / towns["pop_2015"] * 1000
ai_ranking = towns.sort_values("ai_rate", ascending=False)
print(ai_ranking[["town", "ai_rate"]].round(2).to_string(index=False))
ai_top = ai_ranking.iloc[0]
print("Het hoogste cijfer volgens de AI: stad", ai_top["town"], "met", round(ai_top["ai_rate"], 2), "per 1.000")

De code draait, en toch klopt de ranglijst niet. De assistent deelde de gevallen van 2025 door de volkstelling van 2015. Een stad die sinds 2015 groeide, heeft nu meer inwoners dan haar oude basis, dus haar cijfer komt te hoog uit. De snelst gegroeide steden klimmen in de ranglijst van de AI omdat ze groeiden, niet omdat meer van hun inwoners griep kregen.

De volgende cel deelt door de bevolking van 2025. Vergelijk de twee ranglijsten.

In [ ]:
towns["rate"] = towns["cases_2025"] / towns["pop_2025"] * 1000      # gevallen van 2025 gedeeld door inwoners van 2025
ranking = towns.sort_values("rate", ascending=False).reset_index(drop=True)
print(ranking[["town", "rate"]].round(2).to_string(index=False))
top = ranking.iloc[0]
print("Hoogste cijfer met de basis van 2025: stad", top["town"], "met", round(top["rate"], 2), "per 1.000")
place = int(ranking.index[ranking["town"] == ai_top["town"]][0]) + 1
print("Stad", ai_top["town"], "met de basis van 2025:", round(ranking.loc[place - 1, "rate"], 2), "per 1.000, op plaats", place, "van 8")
print("Stad", ai_top["town"], "groeide met", round(change[towns["town"] == ai_top["town"]].iloc[0], 1), "procent sinds de volkstelling")

# CHECK: elk cijfer moet de bevolking van hetzelfde jaar als de gevallen gebruiken
assert np.allclose(towns["rate"], towns["cases_2025"] / towns["pop_2025"] * 1000)
assert top["town"] != ai_top["town"]
print("CHECK geslaagd")

## 4. Aantallen en cijfers geven een andere volgorde

Vergelijk nu de ranglijst op aantal gevallen met de ranglijst op cijfer. Een grote stad kan tegelijk de meeste gevallen en het laagste cijfer hebben. De grafiek zet de twee naast elkaar, met de steden in beide panelen in dezelfde volgorde.

In [ ]:
by_count = towns.sort_values("cases_2025", ascending=False)["town"].tolist()
by_rate = towns.sort_values("rate", ascending=False)["town"].tolist()
print("op aantal gevallen:", by_count)
print("op cijfer:         ", by_rate)
most = towns.loc[towns["cases_2025"].idxmax()]
print("Meeste gevallen: stad", most["town"], "met", int(most["cases_2025"]), "gevallen")
print("Stad", most["town"], "op cijfer: plaats", by_rate.index(most["town"]) + 1, "van 8, met", round(most["rate"], 2), "per 1.000")

In [ ]:
import matplotlib.pyplot as plt

order = towns.sort_values("cases_2025")
fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
left.barh(order["town"], order["cases_2025"])
left.set_title("Griepgevallen, 2025")
left.set_xlabel("gevallen")
left.set_ylabel("stad")
right.barh(order["town"], order["rate"])
right.set_title("Griepgevallen per 1.000 inwoners, 2025")
right.set_xlabel("gevallen per 1.000 inwoners")
plt.tight_layout()
plt.show()

## 5. Jouw beurt: liters per persoon per dag

Een cijfer per persoon vraagt vaak eerst om andere eenheden. De tabel hieronder geeft het water dat vijf verzonnen steden in één jaar kregen, in kubieke meters. Eén kubieke meter is 1.000 liter, en een jaar heeft 365 dagen.

Vraag je AI-assistent om code die een kolom `litres_per_person_per_day` toevoegt en in `highest_use` de stad opslaat met de hoogste waarde. Plak de code in de cel, voer hem uit en voer daarna de CHECK-cel uit. Reken één stad met de hand na voordat je het resultaat vertrouwt.

In [ ]:
rng = np.random.default_rng(66)
water = pd.DataFrame({"town": list("PQRST"), "people": rng.integers(2_000, 40_000, size=5)})
water["m3_per_year"] = np.round(water["people"] * rng.uniform(40, 90, size=5), -2)   # kubieke meters in één jaar
water

In [ ]:
# jouw beurt 1: voeg de kolom litres_per_person_per_day toe en sla in highest_use de stad op met de hoogste waarde
highest_use = None

In [ ]:
# CHECK
assert highest_use is not None, "highest_use is nog leeg"
expected = water["m3_per_year"] * 1000 / water["people"] / 365
assert np.allclose(water["litres_per_person_per_day"], expected), "controleer de eenheden: liters, per persoon, per dag"
assert highest_use == water.loc[expected.idxmax(), "town"]
print("CHECK geslaagd: stad", highest_use, "gebruikt", round(expected.max(), 1), "liter per persoon per dag")

## 6. Een kleine basis geeft uitschieters

De volgende cel maakt 300 dorpen van heel verschillende grootte. Elk dorp heeft hetzelfde ware cijfer van 4 gevallen per 1.000 inwoners per jaar. Alleen het toeval bepaalt hoeveel gevallen elk dorp noteert. Raad voordat je de cel uitvoert welke dorpen de hoogste cijfers laten zien.

In [ ]:
rng = np.random.default_rng(61)
people = np.round(np.exp(rng.uniform(np.log(200), np.log(20_000), size=300))).astype(int)
cases = rng.binomial(people, 0.004)                      # overal hetzelfde ware cijfer: 4 per 1.000
villages = pd.DataFrame({"people": people, "cases": cases})
villages["rate"] = villages["cases"] / villages["people"] * 1000

top10 = villages.sort_values("rate", ascending=False).head(10)
print(top10.round(2).to_string(index=False))
print()
small = villages[villages["people"] < 1000]
large = villages[villages["people"] >= 10_000]
print("Dorpen onder 1.000 inwoners:", len(small), "met cijfers van", round(small["rate"].min(), 2), "tot", round(small["rate"].max(), 2), "per 1.000")
print("Dorpen van 10.000 inwoners of meer:", len(large), "met cijfers van", round(large["rate"].min(), 2), "tot", round(large["rate"].max(), 2), "per 1.000")
print("Van de 10 hoogste cijfers liggen er", int((top10["people"] < 1000).sum()), "in dorpen onder 1.000 inwoners")
print("Eén extra geval telt", 1 / 200 * 1000, "per 1.000 op in een dorp van 200 inwoners")
print("Eén extra geval telt", 1 / 20_000 * 1000, "per 1.000 op in een dorp van 20.000 inwoners")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(villages["people"], villages["rate"], s=12)
ax.axhline(4, color="grey", linewidth=1)
ax.set_xscale("log")
ax.set_title("300 dorpen met hetzelfde ware cijfer van 4 per 1.000 (grijze lijn)")
ax.set_xlabel("inwoners van het dorp (logaritmische schaal)")
ax.set_ylabel("gevallen per 1.000 inwoners")
plt.tight_layout()
plt.show()

De hoogste en de laagste cijfers horen allebei bij kleine dorpen. Eén extra geval verschuift het cijfer van een klein dorp flink en dat van een groot dorp nauwelijks, dus een ranglijst van cijfers zet kleine plaatsen aan beide uiteinden, ook als er niets verschilt.

Als je een cijfer meldt, zet dan het aantal en de basis ernaast, zodat de lezer kan zien hoeveel gewicht het cijfer kan dragen.

## 7. Een verdubbeld risico, in mensen

Een kop zegt dat een gewoonte het risico op een ziekte verdubbelt. Die zin geeft het relatieve risico en verbergt de basis. De cel hieronder neemt twee gevallen: een zeldzame ziekte die van 3 op 10.000 mensen naar 6 op 10.000 gaat, en een veelvoorkomende die van 20 op 100 naar 40 op 100 gaat. Beide risico's verdubbelen. Tel in elk geval de extra mensen.

In [ ]:
cases_risk = [("zeldzame ziekte", 3 / 10_000, 6 / 10_000), ("veelvoorkomende ziekte", 20 / 100, 40 / 100)]
for name, before, after in cases_risk:
    print(f"{name}: relatief risico {after / before:.1f}")
    print(f"  per 10.000 mensen: {before * 10_000:.0f} ervoor en {after * 10_000:.0f} erna")
    print(f"  extra gevallen per 10.000 mensen: {(after - before) * 10_000:.0f}")

Het relatieve risico is in beide gevallen hetzelfde, en het aantal extra mensen verschilt sterk. Een eerlijke versie van de kop geeft het aantal per 1.000 of per 10.000 mensen ervoor en erna.

## 8. Hoe vaak kwam het voor voordat je keek

Een test voor een ziekte vindt 90% van de mensen die de ziekte hebben en wijst 5% van de mensen zonder de ziekte ten onrechte aan. Iemand test positief. Hoe waarschijnlijk is het dat die persoon de ziekte heeft? Het antwoord hangt af van de basisfrequentie: hoe vaak de ziekte voorkwam voordat iemand werd getest.

De functie hieronder telt wat er gebeurt als 10.000 mensen de test doen. Mensen tellen in plaats van percentages vermenigvuldigen heet werken met natuurlijke frequenties. De cel draait dezelfde test bij drie basisfrequenties.

In [ ]:
def screen(base_rate, catches, false_alarm, people=10_000):
    # natuurlijke frequenties: tel wat er gebeurt als zoveel mensen worden getest
    real = people * base_rate
    true_pos = real * catches
    false_pos = (people - real) * false_alarm
    share_real = true_pos / (true_pos + false_pos)
    print(f"basisfrequentie {base_rate:.1%}: {real:.0f} echte gevallen, {true_pos:.0f} terecht positief, "
          f"{false_pos:.0f} fout-positief, {share_real:.1%} van de positieven is echt")
    return share_real

for b in [0.01, 0.10, 0.30]:
    screen(b, catches=0.90, false_alarm=0.05)

Als 1 op de 100 mensen de ziekte heeft, is het grootste deel van de positieve uitslagen vals alarm. De test is in alle drie de regels hetzelfde, en alleen de basisfrequentie verandert. De volgende cel controleert het tellen met een simulatie van 100.000 mensen.

In [ ]:
rng = np.random.default_rng(3)
n = 100_000
ill = rng.random(n) < 0.01
positive = np.where(ill, rng.random(n) < 0.90, rng.random(n) < 0.05)
print("gesimuleerde mensen:", n)
print("positieve uitslagen:", int(positive.sum()))
print("aandeel van de positieven dat ziek is:", round(ill[positive].mean() * 100, 1), "procent")

## 9. Jouw beurt: fraudemeldingen

Dezelfde rekensom geldt overal waar naar iets zeldzaams wordt gezocht, zoals fraude, beveiligingsdreigingen of sollicitaties. De melding van een bank vindt 90% van de frauduleuze betalingen en markeert 1% van de eerlijke betalingen ten onrechte. Twee op de 1.000 betalingen zijn fraude.

Vraag eerst je AI-assistent: *"Welk deel van de gemarkeerde betalingen is fraude?"* Schrijf het antwoord op. Sla daarna zelf het aandeel op in `share_fraud`, als getal tussen 0 en 1, met `screen` of met je eigen code, en voer de CHECK-cel uit.

In [ ]:
# jouw beurt 2: sla in share_fraud het aandeel van de gemarkeerde betalingen op dat fraude is
share_fraud = None

In [ ]:
# CHECK
assert share_fraud is not None, "share_fraud is nog leeg"
expected = 0.002 * 0.90 / (0.002 * 0.90 + 0.998 * 0.01)
assert abs(share_fraud - expected) < 0.001, "tel ook de eerlijke betalingen die gemarkeerd worden"
print("CHECK geslaagd:", round(expected * 100, 1), "procent van de gemarkeerde betalingen is fraude")

## 10. Schrijf het voor een lezer

Schrijf in een tekstcel hieronder twee korte alinea's aan een vriend wiens test uit deel 8 positief was, bij een basisfrequentie van 1 op 100. Gebruik aantallen mensen, geen percentages. Sluit af met wat je vriend nu het beste kan doen.